# 02 — PyTorch Training

End-to-end bacteria classification with PyTorch + `timm`:
1. Load and split data with stratification
2. Build an EfficientNet-B3 feature extractor (ImageNet weights)
3. Train with AdamW + cosine schedule + mixed precision
4. Evaluate on the test set — accuracy, classification report, confusion matrix


In [ ]:
import sys
sys.path.insert(0, '../src')

import torch
import torch.nn as nn
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from tqdm.notebook import tqdm

from bacteria_classifier.dataset import get_dataloaders
from bacteria_classifier.models import build_model
from bacteria_classifier.transforms import get_train_transforms, get_val_transforms
from bacteria_classifier.utils import (
    plot_confusion_matrix, plot_training_curves, print_classification_report
)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
DATA_ROOT = '../data/bacteria'
EPOCHS = 30
BATCH  = 32
LR     = 3e-4

print(f'Device: {DEVICE}')

## 1. Data loaders

In [ ]:
loaders = get_dataloaders(
    root=DATA_ROOT,
    train_transform=get_train_transforms(224),
    val_transform=get_val_transforms(224),
    batch_size=BATCH,
    num_workers=2,
)

NUM_CLASSES  = len(loaders['train'].dataset.dataset.classes)
CLASS_NAMES  = loaders['train'].dataset.dataset.classes

print(f'Classes: {NUM_CLASSES}')
print(f'Train: {len(loaders["train"].dataset)}   Val: {len(loaders["val"].dataset)}   Test: {len(loaders["test"].dataset)}')

## 2. Model

In [ ]:
model = build_model('efficientnet_b3', num_classes=NUM_CLASSES, pretrained=True).to(DEVICE)
criterion  = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer  = AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler  = CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)
scaler     = torch.amp.GradScaler(enabled=DEVICE.type == 'cuda')

## 3. Training loop

In [ ]:
history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}

def run_epoch(loader, train=True):
    if train:
        model.train()
    else:
        model.eval()
    total_loss, correct, total = 0., 0, 0
    preds_all, labels_all = [], []
    ctx = torch.enable_grad() if train else torch.no_grad()
    with ctx:
        for imgs, labels in loader:
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            if train:
                optimizer.zero_grad(set_to_none=True)
            with torch.autocast(DEVICE.type, enabled=DEVICE.type=='cuda'):
                logits = model(imgs)
                loss   = criterion(logits, labels)
            if train:
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(optimizer)
                scaler.update()
            total_loss += loss.item() * imgs.size(0)
            preds = logits.argmax(1)
            correct += (preds == labels).sum().item()
            total   += imgs.size(0)
            preds_all.extend(preds.cpu().tolist())
            labels_all.extend(labels.cpu().tolist())
    return total_loss / total, 100. * correct / total, labels_all, preds_all

best_acc = 0.
for epoch in tqdm(range(1, EPOCHS + 1), desc='Epochs'):
    tr_loss, tr_acc, _, _ = run_epoch(loaders['train'], train=True)
    va_loss, va_acc, _, _ = run_epoch(loaders['val'],   train=False)
    scheduler.step()
    history['train_loss'].append(tr_loss)
    history['val_loss'].append(va_loss)
    history['train_acc'].append(tr_acc)
    history['val_acc'].append(va_acc)
    if va_acc > best_acc:
        best_acc = va_acc
        torch.save(model.state_dict(), '/tmp/best_bacteria.pt')
    if epoch % 5 == 0:
        print(f'  Epoch {epoch:3d}: loss {tr_loss:.4f}/{va_loss:.4f}  acc {tr_acc:.1f}%/{va_acc:.1f}%')

## 4. Training curves

In [ ]:
import matplotlib.pyplot as plt
plot_training_curves(
    history['train_loss'], history['val_loss'],
    history['train_acc'],  history['val_acc']
)
plt.show()

## 5. Test evaluation

In [ ]:
model.load_state_dict(torch.load('/tmp/best_bacteria.pt', map_location=DEVICE, weights_only=True))
_, test_acc, y_true, y_pred = run_epoch(loaders['test'], train=False)

print(f'Test accuracy: {test_acc:.2f}%')
print_classification_report(y_true, y_pred, CLASS_NAMES)

plot_confusion_matrix(y_true, y_pred, CLASS_NAMES, normalize=True)
plt.show()